In [18]:
import os
import glob
import numpy as np
import pandas as pd

folder_path = r"C:\Users\brian\OneDrive\Trading\Market Making\runs\run_20261103_075220"

def add_trade_impact_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Adds future return at multiple time horizons using snapshots.
    Assumes both inputs are already sorted by ts.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        out[f"trade_return_{h}ms"] = (
            (merged[f"future_mid_{h}"] - out["price"]) / out["price"]
        )

    return out

def label_toxicity_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Toxicity = price moved against aggressor after trade.
    Multi-horizon version.
    Assumes sorted inputs.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        move = merged[f"future_mid_{h}"] - out["price"]

        out[f"toxicity_{h}ms"] = np.where(
            out["side"] == "BUY",
            move < 0,
            move > 0
        ).astype(int)

        out[f"signed_impact_{h}ms"] = np.where(
            out["side"] == "BUY",
            -move,
            move
        )

        out[f"signed_impact_bps_{h}ms"] = (
            10000 * out[f"signed_impact_{h}ms"] / out["price"]
        )

    return out

def align_trades_to_snapshots(trades: pd.DataFrame, snapshots: pd.DataFrame):

    snap = snapshots[["ts", "mid", "best_bid", "best_ask"]].rename(columns={
        "mid": "snap_mid",
        "best_bid": "snap_bid",
        "best_ask": "snap_ask"
    })

    merged = pd.merge_asof(trades, snap, on="ts", direction="backward")

    trades = merged.copy()

    trades["snapshot_mid"] = trades["snap_mid"]
    trades["snapshot_bid"] = trades["snap_bid"]
    trades["snapshot_ask"] = trades["snap_ask"]
    trades["spread"] = trades["snap_ask"] - trades["snap_bid"]

    return trades

def label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000)):

    snap_ts = snapshots["ts"].values
    snap_mid = snapshots["mid"].values

    fills = fills.copy()

    base_idx = np.searchsorted(snap_ts, fills["ts"].values, side="right") - 1
    base_idx = np.clip(base_idx, 0, len(snap_ts) - 1)

    fills["snap_mid"] = snap_mid[base_idx]

    fill_ts = fills["ts"].values
    fill_price = fills["price"].values
    
    for h in horizons_ms:
        target = fill_ts + h

        idx = np.searchsorted(snap_ts, target, side="left")
        idx = np.clip(idx, 0, len(snap_ts) - 1)

        # NEW CORRECT SIGNED LOGIC
        # Assume: +1 = buy, -1 = sell
        fills[f"signed_markout_{h}ms"] = fills["fill_sign"] * (snap_mid[idx] - fill_price)
        fills[f"signed_adverse_{h}ms"] = -fills["fill_sign"] * (snap_mid[idx] - fill_price)

    # NEW FOR TOXICITY MODELLING
    snapshots["micro_signal"] = (snapshots["microprice_dev"]) / snapshots["mid"]

    snapshot_cols = [
        "ts",
        "symbol",
        "mid",
        "microprice",
        "microprice_dev",
        "micro_signal",
        "spread",
        "volatility",
        "order_imbalance",
        "trade_imbalance",
        "quote_churn",
        "inventory",
        "regime",
    ]

    snap = snapshots[snapshot_cols].copy()
    snap = snap.rename(columns={"ts": "snapshot_ts"})
    fills = fills.rename(columns={"ts": "fill_ts"})

    fills = pd.merge_asof(
        fills,
        snap,
        left_on="fill_ts",
        right_on="snapshot_ts",
        by="symbol",
        direction="backward",
    )

    fills["snapshot_age_ms"] = fills["fill_ts"] - fills["snapshot_ts"]
    fills = fills.drop(columns=[col for col in fills.columns if col.endswith("_y")])
    fills.columns = fills.columns.str.replace("_x", "", regex=False)

    regime_dummies = pd.get_dummies(fills["regime_id"], prefix="regime").reindex( # one hot encoding for regime toxicity modelling
        columns=["regime_0", "regime_1", "regime_2"], fill_value=0).astype(np.float32)

    fills = pd.concat([fills, regime_dummies], axis=1)

    return fills

def forward_index(ts, horizon):
    n = len(ts)
    j = 0
    idx = np.empty(n, dtype=int)

    for i in range(n):
        while j < n and ts[j] < ts[i] + horizon:
            j += 1
        idx[i] = j if j < n else n

    return idx

def finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000)):

    ts = snapshots["ts"].to_numpy()
    mid = snapshots["mid"].to_numpy()
    n = len(snapshots)

    for h in horizons_ms:
        idx = forward_index(ts, h)

        future_mid = np.full(n, np.nan)
        valid = idx < n

        future_mid[valid] = mid[idx[valid]]
        
        # Store future mid
        snapshots[f"future_mid_{h}ms"] = future_mid

        # Store return
        snapshots[f"future_return_{h}ms"] = (future_mid - mid) / mid

    return snapshots

def compute_utility(market: str, snapshots, lambda_inv=1.0, gamma_tail=1.0):

    df = snapshots.copy()

    pnl = df["total_pnl"].values
    inventory = df["inventory"].values

    # 1. PnL flow (NOT level)
    pnl_flow = np.diff(pnl, prepend=pnl[0])

    pnl_term = np.mean(pnl_flow)

    # 2. Inventory risk (correct time-aligned)
    inv_risk = np.mean(inventory**2)

    # 3. Tail risk (on pnl flow, not level)
    losses = -pnl_flow

    var = np.quantile(losses, 0.95)
    cvar = losses[losses >= var].mean()

    utility = pnl_term - lambda_inv * inv_risk - gamma_tail * cvar

    print("PnL term:", pnl_term)
    print("Inventory penalty:", lambda_inv * inv_risk)
    print("Tail penalty:", gamma_tail * cvar)
    print("Utility:", utility)

    return utility

def load_parquet_chunks(folder_path, pattern, sort_by=None):
    files = glob.glob(os.path.join(folder_path, pattern))

    if not files:
        print(f"Skipping {pattern}: no files found")
        return None

    def chunk_id(path):
        return int(os.path.basename(path).split("_")[-1].split(".")[0])

    files = sorted(files, key=chunk_id)

    df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

    if sort_by and sort_by in df.columns:
        df = df.sort_values(sort_by).reset_index(drop=True)

    print(f"Loaded {pattern}: {len(df):,} rows from {len(files)} files")

    return df

def generate_datasets():

    datasets = {
        "spot_events": ("spot_events_*.parquet", "local_ts"),
        "spot_snapshots": ("spot_snapshots_*.parquet", "ts"),
        "spot_trades": ("spot_trades_*.parquet", "ts"),
        "spot_quotes": ("spot_quotes_*.parquet", "ts"),
        "spot_fills": ("spot_fills_*.parquet", "ts"),

        "perps_events": ("perps_events_*.parquet", "local_ts"),
        "perps_snapshots": ("perps_snapshots_*.parquet", "ts"),
        "perps_trades": ("perps_trades_*.parquet", "ts"),
        "perps_quotes": ("perps_quotes_*.parquet", "ts"),
        "perps_fills": ("perps_fills_*.parquet", "ts"),
    }

    data = {}

    for name, (pattern, sort_by) in datasets.items():
        data[name] = load_parquet_chunks(folder_path, pattern, sort_by)

    # Assign available datasets
    spot_events = data["spot_events"]
    spot_snapshots = data["spot_snapshots"]
    spot_trades = data["spot_trades"]
    spot_quotes = data["spot_quotes"]
    spot_fills = data["spot_fills"]

    perps_events = data["perps_events"]
    perps_snapshots = data["perps_snapshots"]
    perps_trades = data["perps_trades"]
    perps_quotes = data["perps_quotes"]
    perps_fills = data["perps_fills"]

    horizons = (100, 500, 1000, 5000)

    # SPOT
    if spot_snapshots is not None:
        if spot_fills is not None:
            spot_fills = label_fill_markouts_ms(spot_fills, spot_snapshots, horizons)

        if spot_trades is not None:
            spot_trades = add_trade_impact_multi(spot_trades, spot_snapshots, horizons)
            spot_trades = label_toxicity_multi(spot_trades, spot_snapshots, horizons)
            spot_trades = align_trades_to_snapshots(spot_trades, spot_snapshots)

        spot_snapshots = finalize_returns(spot_snapshots, horizons)

        if "total_pnl" in spot_snapshots.columns:
            compute_utility("spot", spot_snapshots)

    # PERPS
    if perps_snapshots is not None:
        if perps_fills is not None:
            perps_fills = label_fill_markouts_ms(perps_fills, perps_snapshots, horizons)

        if perps_trades is not None:
            perps_trades = add_trade_impact_multi(perps_trades, perps_snapshots, horizons)
            perps_trades = label_toxicity_multi(perps_trades, perps_snapshots, horizons)
            perps_trades = align_trades_to_snapshots(perps_trades, perps_snapshots)

        perps_snapshots = finalize_returns(perps_snapshots, horizons)

        if "total_pnl" in perps_snapshots.columns:
            compute_utility("perps", perps_snapshots)

    for name, df in {
        "spot_events": spot_events,
        "spot_snapshots": spot_snapshots,
        "spot_trades": spot_trades,
        "spot_quotes": spot_quotes,
        "spot_fills": spot_fills,

        "perps_events": perps_events,
        "perps_snapshots": perps_snapshots,
        "perps_trades": perps_trades,
        "perps_quotes": perps_quotes,
        "perps_fills": perps_fills,
    }.items():

        if df is not None:
            df.to_parquet(os.path.join(folder_path, f"{name}.parquet"), index=False)

    # Delete original chunk files only after successful processing and saving
    chunk_patterns = [
        "spot_events_*.parquet",
        "spot_snapshots_*.parquet",
        "spot_trades_*.parquet",
        "spot_quotes_*.parquet",
        "spot_fills_*.parquet",

        "perps_events_*.parquet",
        "perps_snapshots_*.parquet",
        "perps_trades_*.parquet",
        "perps_quotes_*.parquet",
        "perps_fills_*.parquet",
    ]

    deleted = 0

    for pattern in chunk_patterns:
        for f in glob.glob(os.path.join(folder_path, pattern)):
            os.remove(f)
            deleted += 1

    print(f"Deleted {deleted} original chunk files.")

generate_datasets()

Loaded spot_events_*.parquet: 496 rows from 3 files
Loaded spot_snapshots_*.parquet: 317 rows from 3 files
Loaded spot_trades_*.parquet: 177 rows from 4 files
Loaded spot_quotes_*.parquet: 4 rows from 4 files
Skipping spot_fills_*.parquet: no files found
Loaded perps_events_*.parquet: 1,774 rows from 9 files
Loaded perps_snapshots_*.parquet: 765 rows from 6 files
Loaded perps_trades_*.parquet: 1,107 rows from 23 files
Loaded perps_quotes_*.parquet: 200 rows from 200 files
Skipping perps_fills_*.parquet: no files found
PnL term: 0.0
Inventory penalty: 3610000000000.0
Tail penalty: 6.012658227848142e-05
Utility: -3610000000000.0
PnL term: 0.0
Inventory penalty: 0.0
Tail penalty: 0.0
Utility: 0.0
Deleted 252 original chunk files.


In [2]:
import os
import glob
import numpy as np
import pandas as pd

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20261023_041050"

def add_trade_impact_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Adds future return at multiple time horizons using snapshots.
    Assumes both inputs are already sorted by ts.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        out[f"trade_return_{h}ms"] = (
            (merged[f"future_mid_{h}"] - out["price"]) / out["price"]
        )

    return out

def label_toxicity_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Toxicity = price moved against aggressor after trade.
    Multi-horizon version.
    Assumes sorted inputs.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        move = merged[f"future_mid_{h}"] - out["price"]

        out[f"toxicity_{h}ms"] = np.where(
            out["side"] == "BUY",
            move < 0,
            move > 0
        ).astype(int)

        out[f"signed_impact_{h}ms"] = np.where(
            out["side"] == "BUY",
            -move,
            move
        )

        out[f"signed_impact_bps_{h}ms"] = (
            10000 * out[f"signed_impact_{h}ms"] / out["price"]
        )

    return out

def align_trades_to_snapshots(trades: pd.DataFrame, snapshots: pd.DataFrame):

    snap = snapshots[["ts", "mid", "best_bid", "best_ask"]].rename(columns={
        "mid": "snap_mid",
        "best_bid": "snap_bid",
        "best_ask": "snap_ask"
    })

    merged = pd.merge_asof(trades, snap, on="ts", direction="backward")

    trades = merged.copy()

    trades["snapshot_mid"] = trades["snap_mid"]
    trades["snapshot_bid"] = trades["snap_bid"]
    trades["snapshot_ask"] = trades["snap_ask"]
    trades["spread"] = trades["snap_ask"] - trades["snap_bid"]

    return trades

def label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000)):

    snap_ts = snapshots["ts"].values
    snap_mid = snapshots["mid"].values

    fills = fills.copy()

    base_idx = np.searchsorted(snap_ts, fills["ts"].values, side="right") - 1
    base_idx = np.clip(base_idx, 0, len(snap_ts) - 1)

    fills["snap_mid"] = snap_mid[base_idx]

    fill_ts = fills["ts"].values
    fill_price = fills["price"].values
    
    for h in horizons_ms:
        target = fill_ts + h

        idx = np.searchsorted(snap_ts, target, side="left")
        idx = np.clip(idx, 0, len(snap_ts) - 1)

        # NEW CORRECT SIGNED LOGIC
        # Assume: +1 = buy, -1 = sell
        fills[f"signed_markout_{h}ms"] = fills["fill_sign"] * (snap_mid[idx] - fill_price)
        fills[f"signed_adverse_{h}ms"] = -fills["fill_sign"] * (snap_mid[idx] - fill_price)

    # NEW FOR TOXICITY MODELLING
    snapshots["micro_signal"] = (snapshots["microprice_dev"]) / snapshots["mid"]

    snapshot_cols = [
        "ts",
        "symbol",
        "mid",
        "microprice",
        "microprice_dev",
        "micro_signal",
        "spread",
        "volatility",
        "order_imbalance",
        "trade_imbalance",
        "quote_churn",
        "inventory",
        "regime",
    ]

    snap = snapshots[snapshot_cols].copy()
    snap = snap.rename(columns={"ts": "snapshot_ts"})
    fills = fills.rename(columns={"ts": "fill_ts"})

    fills = pd.merge_asof(
        fills,
        snap,
        left_on="fill_ts",
        right_on="snapshot_ts",
        by="symbol",
        direction="backward",
    )

    fills["snapshot_age_ms"] = fills["fill_ts"] - fills["snapshot_ts"]
    fills = fills.drop(columns=[col for col in fills.columns if col.endswith("_y")])
    fills.columns = fills.columns.str.replace("_x", "", regex=False)

    regime_dummies = pd.get_dummies(fills["regime_id"], prefix="regime").reindex( # one hot encoding for regime toxicity modelling
        columns=["regime_0", "regime_1", "regime_2"], fill_value=0).astype(np.float32)

    fills = pd.concat([fills, regime_dummies], axis=1)

    return fills

def forward_index(ts, horizon):
    n = len(ts)
    j = 0
    idx = np.empty(n, dtype=int)

    for i in range(n):
        while j < n and ts[j] < ts[i] + horizon:
            j += 1
        idx[i] = j if j < n else n

    return idx

def finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000)):

    ts = snapshots["ts"].to_numpy()
    mid = snapshots["mid"].to_numpy()
    n = len(snapshots)

    for h in horizons_ms:
        idx = forward_index(ts, h)

        future_mid = np.full(n, np.nan)
        valid = idx < n

        future_mid[valid] = mid[idx[valid]]
        
        # Store future mid
        snapshots[f"future_mid_{h}ms"] = future_mid

        # Store return
        snapshots[f"future_return_{h}ms"] = (future_mid - mid) / mid

    return snapshots

def compute_utility(snapshots, lambda_inv=1.0, gamma_tail=1.0):

    df = snapshots.copy()

    pnl = df["total_pnl"].values
    inventory = df["inventory"].values

    # 1. PnL flow (NOT level)
    pnl_flow = np.diff(pnl, prepend=pnl[0])

    pnl_term = np.mean(pnl_flow)

    # 2. Inventory risk (correct time-aligned)
    inv_risk = np.mean(inventory**2)

    # 3. Tail risk (on pnl flow, not level)
    losses = -pnl_flow

    var = np.quantile(losses, 0.95)
    cvar = losses[losses >= var].mean()

    utility = pnl_term - lambda_inv * inv_risk - gamma_tail * cvar

    print("PnL term:", pnl_term)
    print("Inventory penalty:", lambda_inv * inv_risk)
    print("Tail penalty:", gamma_tail * cvar)
    print("Utility:", utility)

    return utility

def generate_datasets():

    def chunk_id(path):
        return int(os.path.basename(path).split("_")[1].split(".")[0])

    events_files = sorted(glob.glob(os.path.join(folder_path, "events_*.parquet")), key = chunk_id)
    events = pd.concat((pd.read_parquet(f) for f in events_files), ignore_index=True)

    snapshots_files = sorted(glob.glob(os.path.join(folder_path, "snapshots_*.parquet")), key = chunk_id)
    snapshots = pd.concat((pd.read_parquet(f) for f in snapshots_files), ignore_index=True)

    trades_files = sorted(glob.glob(os.path.join(folder_path, "trades_*.parquet")), key = chunk_id)
    trades = pd.concat((pd.read_parquet(f) for f in trades_files), ignore_index=True)

    quotes_files = sorted(glob.glob(os.path.join(folder_path, "quotes_*.parquet")), key = chunk_id)
    quotes = pd.concat((pd.read_parquet(f) for f in quotes_files), ignore_index=True)

    fills_files = sorted(glob.glob(os.path.join(folder_path, "fills_*.parquet")), key = chunk_id)
    fills = pd.concat((pd.read_parquet(f) for f in fills_files), ignore_index=True)

    events = events.sort_values("local_ts").copy()
    # events = events.sort_values("ts").copy()
    snapshots = snapshots.sort_values("ts").copy()
    trades = trades.sort_values("ts").copy()
    quotes = quotes.sort_values("ts").copy()
    fills = fills.sort_values("ts").copy()

    # Delete chunk files
    for files in [events_files, snapshots_files, trades_files, quotes_files, fills_files]:
        for f in files:
            os.remove(f)

    fills = label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000))
    
    trades = add_trade_impact_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = label_toxicity_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = align_trades_to_snapshots(trades, snapshots)

    snapshots = finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000))

    compute_utility(snapshots)

    events.to_parquet(os.path.join(folder_path, "events.parquet"), index=False)
    snapshots.to_parquet(os.path.join(folder_path, "snapshots.parquet"), index=False)
    trades.to_parquet(os.path.join(folder_path, "trades.parquet"), index=False)
    quotes.to_parquet(os.path.join(folder_path, "quotes.parquet"), index=False)
    fills.to_parquet(os.path.join(folder_path, "fills.parquet"), index=False)

generate_datasets()

PnL term: -1.6395167127326392e-06
Inventory penalty: 211467816397.74884
Tail penalty: 1.4183728378278419e-05
Utility: -211467816397.74884
